In [21]:
import splitfolders

In [28]:
#Since there are no validation and test masks, I have to divide the train data
splitfolders.ratio('images/Train', output="dataset/images", seed=1337, ratio=(.8, 0.1,0.1))

In [25]:
def ConvertToYOLO(bbox, imgWidth, imgHeight, classId=0, paddingPercent=0.1):
    x_min, y_min, x_max, y_max = bbox
    box_width = x_max - x_min
    box_height = y_max - y_min

    #Padding to save information about skin and optimize segmentation
    pad_x = box_width * paddingPercent
    pad_y = box_height * paddingPercent


    new_x_min = max(0, x_min - pad_x)
    new_y_min = max(0, y_min - pad_y)
    new_x_max = min(imgWidth, x_max + pad_x)
    new_y_max = min(imgHeight, y_max + pad_y)

    new_box_width = new_x_max - new_x_min
    new_box_height = new_y_max - new_y_min

    x_center = new_x_min + (new_box_width / 2.0)
    y_center = new_y_min + (new_box_height / 2.0)

    yolo_x_center = x_center / imgWidth
    yolo_y_center = y_center / imgHeight
    yolo_width = new_box_width / imgWidth
    yolo_height = new_box_height / imgHeight

    #returning string to save in a txt file (YOLO format)
    return f"{classId} {yolo_x_center:.6f} {yolo_y_center:.6f} {yolo_width:.6f} {yolo_height:.6f}"

In [29]:
import os
def CreateLabels(maskPath, labelPath):
    for maskName in os.listdir(maskPath):
        mask=Image.open(os.path.join(maskPath, maskName))
        bbox = mask.getbbox()
        width, height = mask.size
        yoloString=ConvertToYOLO(bbox, width, height)
        base_name, _ = os.path.splitext(maskName)
        newName = base_name + ".txt"
        save_path = os.path.join(labelPath, newName)
        with open(save_path, "w") as f:
            f.write(yoloString)

In [ ]:
#Training bounding boxes
CreateLabels("dataset/images/train/Train_gt", "dataset/labels/train")

In [ ]:
#Testing bounding boxes
CreateLabels("dataset/images/test/Train_gt", "dataset/labels/test")

In [ ]:
#Validation bounding boxes
CreateLabels("dataset/images/val/Train_gt", "dataset/labels/val")

dataset folder was manually altered to achieve folder structure necessary for YOLO detection